# Indian Traffic Sign Recognition (TSR) - YOLOv8n Training Pipeline
### Google Colab Free Cloud GPU Setup (Tesla T4 / L4)

This notebook trains a **YOLOv8n** model for detecting and classifying Indian traffic signs (15 classes).

**Roadmap:**
1. Verify GPU acceleration
2. Mount Google Drive & install Ultralytics
3. Configure `dataset.yaml`
4. Train YOLOv8n (100 epochs, 640x640, AdamW)
5. Evaluate (mAP@50, mAP@50-95, Confusion Matrix)
6. Run inference test on sample frames
7. Export `best.pt` model weights for local deployment

In [ ]:
# Step 1: Check GPU details
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics and dependencies
!pip install --upgrade ultralytics opencv-python matplotlib

In [ ]:
# Step 3: Mount Google Drive (if your dataset is stored on Drive)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 4: Prepare dataset on Colab SSD for maximum I/O speed
# If your dataset is zipped in Google Drive, unzip it to /content/dataset:
# !unzip -q "/content/drive/MyDrive/dataset.zip" -d /content/dataset

# Alternatively, if you uploaded the TrafficSign folder to Drive:
# !cp -r "/content/drive/MyDrive/TrafficSign/dataset" /content/dataset

import os
print("Checking dataset directory...")
if os.path.exists('/content/dataset'):
    print("Dataset found:", os.listdir('/content/dataset'))
else:
    print("Remember to unzip or copy your dataset into /content/dataset")

In [ ]:
# Step 5: Write dataset.yaml configuration
yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 15
names:
  0: Stop
  1: No Entry
  2: Speed Limit 20
  3: Speed Limit 40
  4: Speed Limit 60
  5: Speed Limit 80
  6: Give Way
  7: School Ahead
  8: Pedestrian Crossing
  9: Left Turn
  10: Right Turn
  11: U-Turn
  12: Horn Prohibited
  13: Parking
  14: Dangerous Curve
"""

with open("/content/dataset.yaml", "w") as f:
    f.write(yaml_content.strip())
print("Created /content/dataset.yaml successfully!")

In [ ]:
# Step 6: Initialize YOLOv8n pretrained weights & Train
from ultralytics import YOLO

# Load pretrained nano model
model = YOLO("yolov8n.pt")

# Train for 100 epochs on GPU
results = model.train(
    data="/content/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    device=0,
    project="tsr_colab_run",
    name="yolov8n_indian_tsr",
    save=True,
    plots=True
)

In [ ]:
# Step 7: Evaluate Model Metrics on Validation Set
metrics = model.val()
print(f"Validation Results:")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Step 8: Visualize Training Plots and Confusion Matrix
from IPython.display import Image, display
import glob

results_png = glob.glob("/content/tsr_colab_run/yolov8n_indian_tsr/results.png")
if results_png:
    print("--- Training Progress ---")
    display(Image(results_png[0]))

cm_png = glob.glob("/content/tsr_colab_run/yolov8n_indian_tsr/confusion_matrix.png")
if cm_png:
    print("--- Confusion Matrix ---")
    display(Image(cm_png[0]))

In [ ]:
# Step 9: Save best weights to Google Drive & local download
import shutil
from google.colab import files

best_weights = "/content/tsr_colab_run/yolov8n_indian_tsr/weights/best.pt"
if os.path.exists(best_weights):
    # Backup to Google Drive
    os.makedirs("/content/drive/MyDrive/TSR_Models", exist_ok=True)
    shutil.copy(best_weights, "/content/drive/MyDrive/TSR_Models/best.pt")
    print("Saved weights to Google Drive: /content/drive/MyDrive/TSR_Models/best.pt")
    
    # Download to your local laptop
    files.download(best_weights)
else:
    print("best.pt not found. Check training logs above.")